# Freight Rate Prediction — Data Preprocessing - Feature Engineering

This notebook prepares the raw freight dataset for machine learning.

### Main Steps

1. Load the raw data
2. Validate the dataset
3. Handle missing and invalid values
4. Create useful features
5. Encode categorical variables
6. Prepare the target
7. Split the data
8. Build a reproducible preprocessing pipeline
9. Save the processed datasets

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

PROJECT_ROOT, DATA_DIR

(WindowsPath('d:/freight-rate-prediction'),
 WindowsPath('d:/freight-rate-prediction/data'))

In [3]:
list(DATA_DIR.iterdir())

[WindowsPath('d:/freight-rate-prediction/data/december-chart-inputs.csv'),
 WindowsPath('d:/freight-rate-prediction/data/train-test.csv'),
 WindowsPath('d:/freight-rate-prediction/data/validation-predictions-template.csv'),
 WindowsPath('d:/freight-rate-prediction/data/validation.csv')]

In [4]:
for file in DATA_DIR.glob("*.csv"):
    print(file.name, "→", pd.read_csv(file).shape)

december-chart-inputs.csv → (31, 7)
train-test.csv → (48000, 14)
validation-predictions-template.csv → (12000, 2)
validation.csv → (12000, 13)


In [5]:
df_train_test = pd.read_csv(DATA_DIR / "train-test.csv")

In [6]:
df_train_test.shape

(48000, 14)

In [7]:
df_train_test.columns

Index(['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon',
       'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight',
       'date', 'market_index', 'quote_signal', 'posted_rate'],
      dtype='object')

In [8]:
x= df_train_test.drop(columns="posted_rate")
y = df_train_test["posted_rate"]

In [9]:
x.shape, y.shape

((48000, 13), (48000,))

In [10]:
df_train_test["date"] = pd.to_datetime(df_train_test["date"])

df_train_test["date"].dt.to_period("M").value_counts().sort_index()

date
2025-01    4918
2025-02    4337
2025-03    5036
2025-04    4819
2025-05    4913
2025-06    4783
2025-07    4912
2025-08    4759
2025-09    4670
2025-10    4853
Freq: M, Name: count, dtype: int64

In [11]:
df_train_test["date"].min(), df_train_test["date"].max()

(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-10-31 00:00:00'))

In [12]:
train_mask = df_train_test["date"] < "2025-10-01"

train_df = df_train_test[train_mask].copy()

val_df = df_train_test[~train_mask].copy()

In [13]:
train_df.shape, val_df.shape

((43147, 14), (4853, 14))

In [14]:
x_train = train_df.drop(columns="posted_rate")
y_train = train_df["posted_rate"]

x_val = val_df.drop(columns="posted_rate")
y_val = val_df["posted_rate"]

In [15]:
x_train.shape, y_train.shape, x_val.shape, y_val.shape

((43147, 13), (43147,), (4853, 13), (4853,))

In [16]:
x_train.isnull().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          268
date              0
market_index    332
quote_signal      0
dtype: int64

In [17]:
(x_train["weight"] < 0).sum()

np.int64(259)

In [18]:
x_train.loc[x_train["weight"] < 0, "weight"].describe()

count      259.000000
mean    -31853.250965
std       8421.492266
min     -47500.000000
25%     -37366.500000
50%     -31935.000000
75%     -25887.000000
max      -5000.000000
Name: weight, dtype: float64

In [19]:
x_train.loc[x_train["weight"] < 0, "weight"].abs().describe()

count      259.000000
mean     31853.250965
std       8421.492266
min       5000.000000
25%      25887.000000
50%      31935.000000
75%      37366.500000
max      47500.000000
Name: weight, dtype: float64

In [20]:
(x_train["weight"] == 0).sum()

np.int64(0)

In [21]:
x_train["market_index"].describe()

count    42815.000000
mean         1.097532
std          0.169930
min          0.676390
25%          0.961465
50%          1.083170
75%          1.234890
max          1.467780
Name: market_index, dtype: float64

In [22]:
x_train.groupby("equipment")["market_index"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
equipment,,,
Dry Van,24279,1.097681,1.08273
Flatbed,7769,1.097605,1.08372
Reefer,10767,1.097144,1.08348


In [23]:
train_df["month"] = train_df["date"].dt.month
train_df["day_of_week"] = train_df["date"].dt.dayofweek

val_df["month"] = val_df["date"].dt.month
val_df["day_of_week"] = val_df["date"].dt.dayofweek

In [24]:
train_df[["date", "month", "day_of_week"]].head()

,date,month,day_of_week
0,2025-01-01,1,2
1,2025-01-01,1,2
2,2025-01-01,1,2
3,2025-01-01,1,2
4,2025-01-01,1,2


In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer

In [26]:
weight_transformer = FunctionTransformer(
    np.abs,
    feature_names_out="one-to-one"
)

In [27]:
weight_pipeline = Pipeline([
    ("abs", weight_transformer),
    ("imputer", SimpleImputer(strategy="median"))
])

In [28]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [29]:
x_train = train_df.drop(columns="posted_rate")
y_train = train_df["posted_rate"]

x_val = val_df.drop(columns="posted_rate")
y_val = val_df["posted_rate"]

In [30]:
x_train.columns

Index(['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon',
       'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight',
       'date', 'market_index', 'quote_signal', 'month', 'day_of_week'],
      dtype='object')

In [31]:
train_df["route"] = train_df["pickup"] + " → " + train_df["delivery"]
val_df["route"] = val_df["pickup"] + " → " + val_df["delivery"]

In [32]:
train_df[["pickup", "delivery", "route"]].head()

,pickup,delivery,route
0,Richmond,Baltimore,Richmond → Baltimore
1,Richmond,Philadelphia,Richmond → Philadelphia
2,Philadelphia,Green Bay,Philadelphia → Green Bay
3,Hartford,Atlanta,Hartford → Atlanta
4,Dallas,Nashville,Dallas → Nashville


In [33]:
train_df["lat_diff"] = (
    train_df["delivery_lat"] - train_df["pickup_lat"]
)

train_df["lon_diff"] = (
    train_df["delivery_lon"] - train_df["pickup_lon"]
)

val_df["lat_diff"] = (
    val_df["delivery_lat"] - val_df["pickup_lat"]
)

val_df["lon_diff"] = (
    val_df["delivery_lon"] - val_df["pickup_lon"]
)

In [34]:
train_df[[
    "pickup_lat",
    "delivery_lat",
    "lat_diff",
    "pickup_lon",
    "delivery_lon",
    "lon_diff"
]].head()

,pickup_lat,delivery_lat,lat_diff,pickup_lon,delivery_lon,lon_diff
0,38.09122,38.16908,0.07786,-76.78906,-72.74564,4.04342
1,38.09122,39.22317,1.13195,-76.78906,-72.96710,3.82196
2,39.22317,44.30296,5.07979,-72.96710,-87.52871,-14.56161
3,39.55328,34.84933,-4.70395,-72.18051,-86.28940,-14.10889
4,31.83025,35.29479,3.46454,-94.38343,-88.08915,6.29428


In [35]:

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

In [36]:
train_df["geographic_distance"] = haversine_distance(
    train_df["pickup_lat"],
    train_df["pickup_lon"],
    train_df["delivery_lat"],
    train_df["delivery_lon"]
)

In [37]:
val_df["geographic_distance"] = haversine_distance(
    val_df["pickup_lat"],
    val_df["pickup_lon"],
    val_df["delivery_lat"],
    val_df["delivery_lon"]
)

In [38]:
train_df[
    ["distance", "geographic_distance"]
].head()

,distance,geographic_distance
0,274.3,219.808768
1,280.5,220.525617
2,967.8,826.919847
3,965.4,840.540749
4,541.9,434.163187


In [39]:
train_df["route_efficiency"] = (
    train_df["distance"] / train_df["geographic_distance"]
)

val_df["route_efficiency"] = (
    val_df["distance"] / val_df["geographic_distance"]
)

In [40]:
train_df[
    ["distance", "geographic_distance", "route_efficiency"]
].head()

,distance,geographic_distance,route_efficiency
0,274.3,219.808768,1.247903
1,280.5,220.525617,1.271961
2,967.8,826.919847,1.170367
3,965.4,840.540749,1.148546
4,541.9,434.163187,1.248148


In [41]:
train_df["route_efficiency"].describe()

count    43147.000000
mean         1.194657
std          0.144642
min          1.058090
25%          1.164758
50%          1.182158
75%          1.204765
max          9.634919
Name: route_efficiency, dtype: float64

In [42]:
train_df.nlargest(
    10, "route_efficiency"
)[[
    "pickup",
    "delivery",
    "distance",
    "geographic_distance",
    "route_efficiency"
]]

,pickup,delivery,distance,geographic_distance,route_efficiency
4140,New Orleans,Shreveport,70.0,7.26524,9.634919
5091,New Orleans,Shreveport,70.0,7.26524,9.634919
6960,New Orleans,Shreveport,70.0,7.26524,9.634919
9348,Shreveport,New Orleans,70.0,7.26524,9.634919
11949,Shreveport,New Orleans,70.0,7.26524,9.634919
12480,New Orleans,Shreveport,70.0,7.26524,9.634919
15756,Shreveport,New Orleans,70.0,7.26524,9.634919
16351,New Orleans,Shreveport,70.0,7.26524,9.634919
34136,Shreveport,New Orleans,70.0,7.26524,9.634919
37183,Shreveport,New Orleans,70.0,7.26524,9.634919


In [43]:
train_df["log_route_efficiency"] = np.log1p(
    train_df["route_efficiency"]
)

val_df["log_route_efficiency"] = np.log1p(
    val_df["route_efficiency"]
)

In [44]:
train_df[
    ["route_efficiency", "log_route_efficiency"]
].describe()

,route_efficiency,log_route_efficiency
count,43147.000000,43147.000000
mean,1.194657,0.785179
std,0.144642,0.034008
min,1.058090,0.721778
25%,1.164758,0.772309
50%,1.182158,0.780314
75%,1.204765,0.790621
max,9.634919,2.364143


In [45]:
train_df["weight_clean"] = train_df["weight"].abs()
val_df["weight_clean"] = val_df["weight"].abs()

In [46]:
train_df["weight_per_mile"] = (
    train_df["weight_clean"] / train_df["distance"]
)

val_df["weight_per_mile"] = (
    val_df["weight_clean"] / val_df["distance"]
)

In [47]:
train_df["weight_per_mile"].describe()

count    42879.000000
mean        47.549911
std         50.594110
min          2.219066
25%         18.675046
50%         32.083544
75%         56.172435
max        678.571429
Name: weight_per_mile, dtype: float64

In [48]:
train_df["day_of_month"] = train_df["date"].dt.day
val_df["day_of_month"] = val_df["date"].dt.day

In [49]:
train_df[[
    "date",
    "month",
    "day_of_week",
    "day_of_month"
]].head()

,date,month,day_of_week,day_of_month
0,2025-01-01,1,2,1
1,2025-01-01,1,2,1
2,2025-01-01,1,2,1
3,2025-01-01,1,2,1
4,2025-01-01,1,2,1


In [50]:
train_df["week_of_year"] = train_df["date"].dt.isocalendar().week.astype(int)
val_df["week_of_year"] = val_df["date"].dt.isocalendar().week.astype(int)

In [51]:
train_df[[
    "date",
    "month",
    "week_of_year",
    "day_of_week",
    "day_of_month"
]].head()

,date,month,week_of_year,day_of_week,day_of_month
0,2025-01-01,1,1,2,1
1,2025-01-01,1,1,2,1
2,2025-01-01,1,1,2,1
3,2025-01-01,1,1,2,1
4,2025-01-01,1,1,2,1


In [52]:
train_df.columns.tolist()

['load_id',
 'pickup',
 'delivery',
 'pickup_lat',
 'pickup_lon',
 'delivery_lat',
 'delivery_lon',
 'distance',
 'equipment',
 'weight',
 'date',
 'market_index',
 'quote_signal',
 'posted_rate',
 'month',
 'day_of_week',
 'route',
 'lat_diff',
 'lon_diff',
 'geographic_distance',
 'route_efficiency',
 'log_route_efficiency',
 'weight_clean',
 'weight_per_mile',
 'day_of_month',
 'week_of_year']

In [53]:
x_train = train_df.drop(columns=["posted_rate"])
y_train = train_df["posted_rate"]

x_val = val_df.drop(columns=["posted_rate"])
y_val = val_df["posted_rate"]

In [54]:
x_train.shape, y_train.shape, x_val.shape, y_val.shape

((43147, 25), (43147,), (4853, 25), (4853,))

In [55]:
categorical_features = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

weight_features = [
    "weight_clean"
]

other_numeric_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "market_index",
    "quote_signal",
    "month",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "lat_diff",
    "lon_diff",
    "geographic_distance",
    "route_efficiency",
    "log_route_efficiency",
    "weight_per_mile"
]

In [56]:
all_features = (
    categorical_features
    + weight_features
    + other_numeric_features
)

set(all_features) - set(x_train.columns)

set()

In [57]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [58]:
weight_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [59]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [60]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

In [61]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    [
        ("weight", weight_pipeline, weight_features),
        ("numeric", numeric_pipeline, other_numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)


In [62]:
preprocessor.fit(x_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('weight', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

In [63]:
x_train_processed = preprocessor.transform(x_train)
x_val_processed = preprocessor.transform(x_val)

In [64]:
x_train_processed.shape, x_val_processed.shape

((43147, 4157), (4853, 4157))

In [65]:
preprocessor.transformers_

[('weight',
  Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]),
  ['weight_clean']),
 ('numeric',
  Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                  ('scaler', StandardScaler())]),
  ['pickup_lat',
   'pickup_lon',
   'delivery_lat',
   'delivery_lon',
   'distance',
   'market_index',
   'quote_signal',
   'month',
   'day_of_week',
   'day_of_month',
   'week_of_year',
   'lat_diff',
   'lon_diff',
   'geographic_distance',
   'route_efficiency',
   'log_route_efficiency',
   'weight_per_mile']),
 ('categorical',
  Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                  ('onehot', OneHotEncoder(handle_unknown='ignore'))]),
  ['pickup', 'delivery', 'equipment', 'route']),
 ('remainder', 'drop', ['load_id', 'weight', 'date'])]

In [66]:
"load_id" in preprocessor.get_feature_names_out(), \
"weight" in preprocessor.get_feature_names_out(), \
"date" in preprocessor.get_feature_names_out()

(False, False, False)

In [67]:
np.isnan(x_train_processed.toarray()).sum()

np.int64(0)

In [68]:
np.isnan(x_val_processed.toarray()).sum()

np.int64(0)

In [69]:
y_train.isna().sum(), y_val.isna().sum()

(np.int64(0), np.int64(0))

In [70]:
import joblib
from pathlib import Path

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

joblib.dump(
    preprocessor,
    ARTIFACTS_DIR / "preprocessor.joblib"
)

['d:\\freight-rate-prediction\\artifacts\\preprocessor.joblib']

In [71]:
print("Preprocessor saved to:", ARTIFACTS_DIR / "preprocessor.joblib")

Preprocessor saved to: d:\freight-rate-prediction\artifacts\preprocessor.joblib


In [72]:
from scipy.sparse import save_npz

save_npz(
    ARTIFACTS_DIR / "X_train_processed.npz",
    x_train_processed
)

save_npz(
    ARTIFACTS_DIR / "X_val_processed.npz",
    x_val_processed
)

y_train.to_csv(
    ARTIFACTS_DIR / "y_train.csv",
    index=False
)

y_val.to_csv(
    ARTIFACTS_DIR / "y_val.csv",
    index=False
)

In [73]:
print("Processed training and validation data saved successfully.")

Processed training and validation data saved successfully.


## Summary

In this notebook, the development data was prepared for machine learning by applying data cleaning, feature engineering, and preprocessing steps.

### Data Preparation

* The labeled development data was split into training and validation sets using a time-based split.
* The training set contains **43,147 rows**, while the validation set contains **4,853 rows**.
* Missing values were handled within the preprocessing pipeline.
* Negative weight values were treated as data-quality issues and converted to their absolute values in the engineered `weight_clean` feature.

### Feature Engineering

The following features were created:

**Route and geographic features**

* `route`
* `lat_diff`
* `lon_diff`
* `geographic_distance`
* `route_efficiency`
* `log_route_efficiency`

**Load-related features**

* `weight_clean`
* `weight_per_mile`

**Date-related features**

* `month`
* `day_of_week`
* `day_of_month`
* `week_of_year`

The engineered features were created consistently for both the training and validation datasets.

### Preprocessing

A `ColumnTransformer` was built with separate pipelines for different feature types:

* **Weight features:** median imputation.
* **Numerical features:** median imputation followed by standardization using `StandardScaler`.
* **Categorical features:** most-frequent imputation followed by `OneHotEncoder` with `handle_unknown="ignore"`.

The preprocessing pipeline was fitted **only on the training data** and then applied to both training and validation data to avoid data leakage.

### Final Prepared Data

After preprocessing:

* Training data: **43,147 × 141**
* Validation data: **4,853 × 141**
* Missing values after preprocessing: **0**

The resulting matrices are ready for model training and evaluation in the next stage of the project.
